# 01- Preprocessing


In [1]:
import tensorflow as tf

import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import re
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from nltk.corpus import stopwords

import sys
sys.path.append('../cs16')
import cs16.prep as prep16
import cs16.plot as plot16
import cs16.build as build16
imagesize = 64
import time
import pandas as pd

import cs16.prep as prep16
import cs16.plot as plot16
import cv2
import numpy as np
import pandas as pd
import os
from keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.regularizers import l2
from keras.preprocessing.text import Tokenizer
from keras import regularizers
from keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
max_words=10000
max_len=100
imagesize =64

#data_type = input("Enter '1. single' for single dataset or '2. multi' for multiple dataset: ")
data_type = '1'
if data_type == '1':
    file_path = '../single.txt'
    folder_path = '../data/MVSA/single/'
elif data_type == '2':
    file_path = '../multi.txt'
    folder_path = '../data/MVSA/multiple/'
else:
    print("Invalid input. Please enter either 'single' or 'multi'.")
    exit()

df = pd.read_csv(file_path, index_col=None, encoding='ISO-8859-1')

### Natural Language Processing.

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import pandas as pd
import matplotlib.pyplot as plt

## It removes URLs, punctuation, and stopwords, converts text to lowercase, tokenizes and 
## stems words, then replaces each tweet in the dataset with its cleaned version.

# Download stopwords and punkt tokenizer if not already downloaded
nltk.download('stopwords')
nltk.download('punkt')

# Define a function to preprocess text
def nlp_text(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Remove URLs using regex
    text = re.sub(r'http\S+', '', text)
    
    # Remove punctuation using regex
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenize text into individual words
    words = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    
    # Stem words using Porter Stemmer
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    
    # Join words back into a single string
    text = ' '.join(words)
    
    return text

# Apply the preprocess_text function to the 'tweet' column of the dataframe
df['tweet'] = df['tweet'].apply(nlp_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ausco\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ausco\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


##### Training, Validation, and Test sets
Preprocessing text and image data 

In [3]:

X_text, y_text = prep16.preprocess_text(df)
X_train_text, X_val_text, X_test_text, \
y_train_text, y_val_text, y_test_text = prep16.split_data(X_text, y_text, random_state=42)

X_polar, y_polar = prep16.preprocess_text(df,label = 'polarity')
X_train_polar, X_val_polar, X_test_polar, \
y_train_polar, y_val_polar, y_test_polar = prep16.split_data(X_polar, y_polar, random_state=42)

image_data_s, image_label_s = prep16.preprocess_images(df, folder_path, imagesize)
y_s = to_categorical(image_label_s, num_classes=3)

X_train_image, X_val_image, X_test_image, \
y_train_image, y_val_image, y_test_image= prep16.split_data(image_data_s, y_s, random_state=42)

y_train = to_categorical(y_train_polar, num_classes=3)
y_val =to_categorical(y_val_polar, num_classes=3)
y_test =to_categorical(y_test_polar, num_classes=3)
file_path

'../single.txt'

### Save preprocessed multimodal Data

In [4]:
import pickle
import os
from datetime import datetime
# This function serializes all preprocessed text, polarity, and image splits, together with 
# metadata, into a single pickle file so the DMS datasets can be reloaded without rerunning 
# preprocessing.
def save_preprocessed_data(save_path='preprocessed_data', filename='single_data.pkl'):
    """
    Save all preprocessed data using pickle
    
    Parameters:
    -----------
    save_path : str
        Directory to save the file
    filename : str
        Name of the pickle file
    """
    
    # Create directory if not exists
    os.makedirs(save_path, exist_ok=True)
    
    # Prepare data dictionary
    data = {
        # Metadata
        'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'description': 'Preprocessed text and image data for DMS',
        
        # Text data (polarity)
        'X_train_polar': X_train_polar,
        'X_val_polar': X_val_polar,
        'X_test_polar': X_test_polar,
        'y_train_polar': y_train_polar,
        'y_val_polar': y_val_polar,
        'y_test_polar': y_test_polar,
        
        # Text data (original)
        'X_train_text': X_train_text,
        'X_val_text': X_val_text,
        'X_test_text': X_test_text,
        'y_train_text': y_train_text,
        'y_val_text': y_val_text,
        'y_test_text': y_test_text,
        
        # Image data
        'X_train_image': X_train_image,
        'X_val_image': X_val_image,
        'X_test_image': X_test_image,
        'y_train_image': y_train_image,
        'y_val_image': y_val_image,
        'y_test_image': y_test_image,
        
        # One-hot encoded labels
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test,
        
        # Additional info
        'num_classes': 3,
        'image_size': imagesize,
        'random_state': 42
    }
    
    # Save to pickle
    filepath = os.path.join(save_path, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    
    print(f"✅ Data saved to {filepath}")
    print(f"   Size: {os.path.getsize(filepath) / 1024 / 1024:.2f} MB")
    print(f"   Timestamp: {data['timestamp']}")
    
    # Optional: Save a small info file
    info_file = os.path.join(save_path, 'data_info.txt')
    with open(info_file, 'w') as f:
        f.write(f"Data saved: {data['timestamp']}\n")
        f.write(f"File: {filename}\n")
        f.write(f"Num classes: {data['num_classes']}\n")
        f.write(f"Image size: {data['image_size']}\n")
        f.write("\nData shapes:\n")
        f.write(f"X_train_polar: {X_train_polar.shape}\n")
        f.write(f"X_train_image: {X_train_image.shape}\n")
    
    return data

In [6]:
# Single
if data_type == '1':
    data = save_preprocessed_data(
        save_path='cache',
        filename='mvsa-single.pkl' )
elif data_type == '2':
    data = save_preprocessed_data(
    save_path='cache',
    filename='mvsa-multiple.pkl')
else:
    print("Invalid input. Please enter either 'single' or 'multi'.")
    exit()


✅ Data saved to cache\mvsa-single.pkl
   Size: 236.00 MB
   Timestamp: 2026-04-15 14:57:48


# ML Feature Extraction

In [7]:
from sklearn.ensemble import RandomForestClassifier # Changed from ExtraTreesClassifier
import time
import numpy as np

start_time = time.time()
#------------------------------------------------------------

########################################
# 1. Label Conversion
# Convert one-hot encoded labels to integer labels for Scikit-Learn compatibility
########################################
y_train_int = np.argmax(y_train, axis=1)
y_val_int   = np.argmax(y_val,   axis=1)
y_test_int  = np.argmax(y_test,  axis=1)

########################################
# 2. Text Modality: Random Forest Classifier
# Generate 3-dimensional probability vectors [p_neg, p_neu, p_pos]
########################################
# Initializing Random Forest for text features
rf_text = RandomForestClassifier(n_estimators=100, random_state=42)

# Training the text model
rf_text.fit(X_train_text, y_train_int)

# Extract class probabilities to use as features for the next stage
train_probs_text = rf_text.predict_proba(X_train_text)  # shape -> (n_train, 3)
val_probs_text   = rf_text.predict_proba(X_val_text)    # shape -> (n_val,   3)
test_probs_text  = rf_text.predict_proba(X_test_text)   # shape -> (n_test,  3)

########################################
# 3. Image Modality: Random Forest Classifier
########################################
# Flatten image data from (N, H, W, C) to (N, D) for the Random Forest input
X_train_img_flat = X_train_image.reshape(X_train_image.shape[0], -1)
X_val_img_flat   = X_val_image.reshape(X_val_image.shape[0],   -1)
X_test_img_flat  = X_test_image.reshape(X_test_image.shape[0], -1)

# Initializing Random Forest for image features with specific hyperparameters
rf_image = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=6,
    max_features='sqrt',
    class_weight='balanced', # Handles potential class imbalance
    random_state=42,
    bootstrap=True           # Traditional RF uses bagging (bootstrap sampling)
)

# Training the image model
rf_image.fit(X_train_img_flat, y_train_int)

# Extract class probabilities for image modality
train_probs_image = rf_image.predict_proba(X_train_img_flat)  # (n_train, 3)
val_probs_image   = rf_image.predict_proba(X_val_img_flat)    # (n_val,   3)
test_probs_image  = rf_image.predict_proba(X_test_img_flat)   # (n_test,  3)

########################################
# 4. Global Fusion: Probability Concatenation
# Concatenate Text(3-dim) + Image(3-dim) to create a 6-dimensional feature vector
########################################
beta = 1
topk = 2
delta = 1e-2

# Combine the outputs of both modalities into a single feature set for global fusion
X_train_global = np.concatenate([train_probs_text, train_probs_image], axis=1)
X_val_global   = np.concatenate([val_probs_text,   val_probs_image], axis=1)
X_test_global  = np.concatenate([test_probs_text,  test_probs_image], axis=1)

print("Dynamic Fusion Feature shape (train):", X_train_global.shape)    
#------------------------------------------------------------

end_time = time.time()
elapsed_time = end_time - start_time

# Display execution time
print(f"Time Executed ===============>：{elapsed_time:.4f} Sec")

Dynamic Fusion Feature shape (train): (3895, 6)
Time Executed ===============>：27.7370 Sec


# Save ML extracted feature

In [10]:
results = {
    # labels
    "y_train_int": y_train_int,
    "y_val_int": y_val_int,
    "y_test_int": y_test_int,
    
    # unimodal probabilities
    "train_probs_text": train_probs_text,
    "val_probs_text": val_probs_text,
    "test_probs_text": test_probs_text,
    
    "train_probs_image": train_probs_image,
    "val_probs_image": val_probs_image,
    "test_probs_image": test_probs_image,
    
    # DPF fused features
    "X_train_global": X_train_global,
    "X_val_global": X_val_global,
    "X_test_global": X_test_global,
    
    # metadata (optional but helpful)
    "n_classes": 3,
    "dataset": "MVSA-Single" if data_type == '1' else "MVSA-Multiple",
    "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

import pickle
import os
from datetime import datetime

os.makedirs("cache", exist_ok=True)

if data_type == '1':
    with open("cache/mvsa-single-ml.pkl", "wb") as f:
        pickle.dump(results, f)
    print("✅ MVSA-Single data saved")
elif data_type == '2': 
    with open("cache/mvsa-multiple-ml.pkl", "wb") as f:
        pickle.dump(results, f)
    print("✅ MVSA-Multiple data saved")
else:
    print("Invalid input. Please enter either '1' or '2'.")
    exit()

# Optional: print file size
import os
file_path = f"cache/mvsa-{'single' if data_type == '1' else 'multiple'}-ml.pkl"
size_mb = os.path.getsize(file_path) / 1024 / 1024
print(f"File size: {size_mb:.2f} MB")

✅ MVSA-Single data saved
File size: 0.48 MB


## ========================================

In [12]:
import pickle

with open("cache/mvsa-single-ml.pkl", "rb") as f:
    results = pickle.load(f)

# RETRIEVE  Varibles from results
train_probs_text   = results["train_probs_text"]
train_probs_image  = results["train_probs_image"]

val_probs_text     = results["val_probs_text"]
val_probs_image    = results["val_probs_image"]

X_train_global = results["X_train_global"]
X_val_global = results["X_val_global"]
X_test_global = results["X_test_global"]


test_probs_text    = results["test_probs_text"]
test_probs_image   = results["test_probs_image"]

y_train_int = results["y_train_int"]
y_val_int   = results["y_val_int"]
y_test_int  = results["y_test_int"]


<div id="Advisor" style="background-color: lightblue; padding: 10px;">
    <h1> Classifiers </h1>
    </div>

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

### Traditional Classifier - XGB

In [32]:
global_clf = xgb.XGBClassifier(n_estimators=100, max_depth=2, random_state=42)
global_clf.fit(X_train_global, y_train_int)

# TEST set
test_pred = global_clf.predict(X_test_global)
print("Test Accuracy:", accuracy_score(y_test_int, test_pred))
print("Test Classification Report:")
print(classification_report(y_test_int, test_pred,
                            digits=4,
                            target_names=['negative','neutral','positive']))

Test Accuracy: 0.5708418891170431
Test Classification Report:
              precision    recall  f1-score   support

    negative     0.5000    0.1039    0.1720        77
     neutral     0.5995    0.8777    0.7124       278
    positive     0.4062    0.1970    0.2653       132

    accuracy                         0.5708       487
   macro avg     0.5019    0.3929    0.3833       487
weighted avg     0.5314    0.5708    0.5058       487



### Logistic Regression

In [34]:
from sklearn.linear_model import LogisticRegression

# LR
global_clf = LogisticRegression(random_state=42, max_iter=100)
global_clf.fit(X_train_global, y_train_int)

# TEST set
test_pred = global_clf.predict(X_test_global)
print("Test Accuracy:", accuracy_score(y_test_int, test_pred))
print("Test Classification Report:")
print(classification_report(y_test_int, test_pred,
                            digits=4,
                            target_names=['negative','neutral','positive']))

Test Accuracy: 0.5790554414784395
Test Classification Report:
              precision    recall  f1-score   support

    negative     0.4545    0.0649    0.1136        77
     neutral     0.6034    0.8921    0.7199       278
    positive     0.4462    0.2197    0.2944       132

    accuracy                         0.5791       487
   macro avg     0.5014    0.3922    0.3760       487
weighted avg     0.5372    0.5791    0.5087       487

